# 36. Efficient Self-Attention 이해

이 노트북은 `35_MiT_Mix_Transformer_Encoder_구조.ipynb` 다음 단계로, SegFormer의 MiT encoder가 self-attention 계산량을 줄이는 방식을 이해합니다.

일반 self-attention은 token 수가 많아질수록 `N x N` attention matrix를 계산해야 합니다. Segmentation은 고해상도 feature가 중요하기 때문에 attention 계산량을 줄이는 설계가 필요합니다.

이번 노트북의 목표는 다음과 같습니다.

- 일반 self-attention의 계산량 부담을 확인합니다.
- sequence reduction이 key/value token 수를 줄이는 방식을 이해합니다.
- stage별 reduction ratio가 왜 달라지는지 감각을 잡습니다.
- SegFormer의 효율성이 segmentation에 중요한 이유를 정리합니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=3, suppress=True)

## 36-1. 일반 self-attention의 token pair 수

feature map이 `H x W`라면 token 수는 `N = H * W`입니다. 일반 attention은 query token마다 모든 key token을 보므로 pair 수가 `N^2`입니다.

In [ ]:
resolutions = np.array([56, 28, 14, 7])
tokens = resolutions ** 2
global_pairs = tokens ** 2

for r, n, p in zip(resolutions, tokens, global_pairs):
    print(f'{r}x{r}: tokens={n:,}, attention pairs={p:,}')

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(tokens, global_pairs, marker='o')
plt.xlabel('token count N')
plt.ylabel('N x N pairs')
plt.title('Global self-attention 계산량 증가')
plt.grid(alpha=0.3)
plt.show()

## 36-2. Sequence reduction attention

SegFormer의 efficient self-attention은 query는 원래 token을 쓰되, key/value 쪽 sequence를 줄입니다.

```text
Q: N tokens
K, V: N / R tokens
attention matrix: N x (N / R)
```

여기서 `R`은 reduction ratio입니다. 고해상도 stage에서는 더 큰 reduction ratio를 사용해 계산량을 줄입니다.

In [ ]:
reduction_ratios = np.array([64, 16, 4, 1])
efficient_pairs = tokens * (tokens // reduction_ratios)

for i, (r, n, rr, ep) in enumerate(zip(resolutions, tokens, reduction_ratios, efficient_pairs), start=1):
    print(f'stage {i}: {r}x{r}, N={n:,}, R={rr}, pairs={ep:,}')

In [ ]:
x = np.arange(len(resolutions))
plt.figure(figsize=(8, 3.8))
plt.bar(x - 0.18, global_pairs, width=0.36, label='global attention')
plt.bar(x + 0.18, efficient_pairs, width=0.36, label='efficient attention')
plt.xticks(x, [f'{r}x{r}' for r in resolutions])
plt.yscale('log')
plt.ylabel('attention pair count, log scale')
plt.title('Sequence reduction으로 줄어드는 attention 계산량')
plt.legend()
plt.show()

## 36-3. 작은 예제로 sequence reduction 보기

아래 예제는 `8 x 8` token grid에서 key/value token을 `2 x 2` block 평균으로 줄이는 과정을 단순화한 것입니다. 실제 구현은 convolution이나 reshape를 통해 더 효율적으로 처리합니다.

In [ ]:
grid = np.arange(8 * 8).reshape(8, 8)
reduced = grid.reshape(4, 2, 4, 2).mean(axis=(1, 3))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, data, title in zip(axes, [grid, reduced], ['original K/V token grid', 'reduced K/V grid']):
    ax.imshow(data, cmap='viridis')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    for y in range(data.shape[0]):
        for x_pos in range(data.shape[1]):
            ax.text(x_pos, y, f'{data[y, x_pos]:.0f}', ha='center', va='center', color='white', fontsize=8)
plt.tight_layout()
plt.show()

## 정리

- 일반 self-attention은 token 수가 많아지면 계산량이 `N^2`로 커집니다.
- SegFormer의 efficient self-attention은 key/value sequence를 줄여 `N x N/R` 형태로 계산합니다.
- 고해상도 stage에서 reduction ratio를 크게 두면 segmentation에 필요한 feature를 유지하면서 계산량을 줄일 수 있습니다.
- 다음 노트북 `37_SegFormer_MLP_Decoder_구조.ipynb`에서는 encoder feature를 mask로 바꾸는 decoder를 살펴봅니다.